# Transfer Learning with ResNet-18 on CIFAR-10

Compare three image classification approaches using PyTorch:
- Training ResNet-18 from scratch
- Fine-tuning an ImageNet-pretrained ResNet-18
- Using a pretrained ResNet-18 as a fixed feature extractor (linear probing)


## 1. Preparing the CIFAR-10 Dataset


#### Loading and Preprocessing CIFAR-10


In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, models, transforms
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor
from torch.utils.data.dataloader import DataLoader

data_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

dataset =  CIFAR10(root='.', train=True, download=True, transform=data_transforms)
test_dataset =  CIFAR10(root='.', train=False, download=True, transform=data_transforms)
class_names = classes = dataset.classes
print(class_names)
"""
class_count = {}
for _, index in dataset:
    label = classes[index]
    if label not in class_count:
        class_count[label] = 0
    class_count[label] += 1
class_count
"""

100%|██████████| 170M/170M [45:42<00:00, 62.2kB/s]


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


'\nclass_count = {}\nfor _, index in dataset:\n    label = classes[index]\n    if label not in class_count:\n        class_count[label] = 0\n    class_count[label] += 1\nclass_count\n'

#### Splitting Training, Validation, and Test Data


In [2]:
from torch.utils.data import random_split

torch.manual_seed(1009)
val_size = 100
train_size = 1000
obsolete_size = len(dataset) - train_size - val_size
#train_size = len(dataset) - val_size

train_ds, val_ds, _ = random_split(dataset, [train_size, val_size, obsolete_size])
#train_ds, val_ds = random_split(dataset, [train_size, val_size])

test_size = 200
test_obsolete_size = (len(test_dataset) - test_size)
test_ds, _ = random_split(test_dataset, [test_size, test_obsolete_size])
"""
torch.manual_seed(1009)
val_size = 5000
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])
test_ds = test_dataset
"""
print(len(train_ds), len(val_ds), len(test_ds))

1000 100 200


#### Checking Class Distribution


In [3]:
class_count = {}
for _, index in train_ds:
    label = classes[index]
    if label not in class_count:
        class_count[label] = 0
    class_count[label] += 1
class_count

{'airplane': 95,
 'horse': 93,
 'bird': 96,
 'ship': 99,
 'frog': 94,
 'dog': 95,
 'cat': 97,
 'automobile': 109,
 'truck': 118,
 'deer': 104}

#### Creating DataLoaders and Selecting the Device


In [4]:
batch_size=128
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size*2, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size*2, num_workers=0, pin_memory=False)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


## 2. Training and Validation Functions


#### Importing Training Utilities


In [5]:

import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import time
import os
from tempfile import TemporaryDirectory
cudnn.benchmark = True

#### Defining the Validation Function


In [6]:
@torch.no_grad()
def validate_model(model, data_loader):
  true_false = []
  model.eval()
  for i, (inputs, labels) in enumerate(data_loader):
    inputs = inputs.to(device)
    labels = labels.to(device)
    outputs = model(inputs)
    preds = torch.argmax(outputs, dim=1)
    true_false.append(preds == labels)
  true_false_tensor = torch.cat(true_false)
  acc = torch.mean(true_false_tensor.float()) * 100
  return acc.item()

#### Defining the Model Training Function


In [7]:
from tqdm.auto import trange, tqdm

def train_model(model, criterion, optimizer, trainloader, validateloader, scheduler, num_epochs=25, val_freq=1, device='cpu'):
    since = time.time()

    # Create a temporary directory to save training checkpoints
    with TemporaryDirectory() as tempdir:
        best_model_params_path = os.path.join(tempdir, 'best_model_params.pt')

        torch.save(model.state_dict(), best_model_params_path)
        best_acc = 0.0
        model.to(device)

        for epoch in trange(num_epochs):
            model.train()
            total_loss = 0.0

            for inputs, labels in tqdm(trainloader, total=len(trainloader)):
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward and loss
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # Backward and update
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * inputs.size(0)

            scheduler.step()
            epoch_loss = total_loss / len(trainloader.dataset)

            if (epoch + 1) % val_freq == 0:
                train_acc = validate_model(model, trainloader)
                val_acc = validate_model(model, validateloader)
                print(f'Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.4f}% | Val Acc: {val_acc:.4f}%')

                if val_acc > best_acc:
                    best_acc = val_acc
                    torch.save(model.state_dict(), best_model_params_path)

        time_elapsed = time.time() - since
        print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
        print(f'Best val Acc: {best_acc:.4f}%')

        # Load the best model weights
        model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
    return model

## 3. Comparing Transfer Learning Methods


### 3.1 Training ResNet-18 from Scratch


#### Initializing the Scratch Model


In [8]:
model_scratch = models.resnet18(weights=None)
in_features = model_scratch.fc.in_features
num_classes = len(train_ds.dataset.classes)
model_scratch.fc = nn.Linear(in_features, num_classes)
criterion = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer_scratch = optim.Adam(model_scratch.parameters(), lr=0.01)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_scratch, step_size=7, gamma=0.1)

#### Training the Scratch Model


In [9]:
model_scratch = train_model(model=model_scratch,
                       criterion = criterion,
                       trainloader = train_loader,
                       validateloader = val_loader,
                       optimizer=optimizer_scratch,
                       scheduler = exp_lr_scheduler,
                       num_epochs=25,
                       device=device)

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 1 | Loss: 3.3558 | Train Acc: 11.8000% | Val Acc: 11.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 2 | Loss: 2.5809 | Train Acc: 10.9000% | Val Acc: 11.0000%


...

Epoch 23 | Loss: 1.7455 | Train Acc: 37.7000% | Val Acc: 29.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 24 | Loss: 1.7471 | Train Acc: 37.1000% | Val Acc: 28.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 25 | Loss: 1.7272 | Train Acc: 37.6000% | Val Acc: 28.0000%
Training complete in 3m 20s
Best val Acc: 29.0000%


#### Evaluating the Scratch Model


In [10]:
test_acc_scratch = validate_model(model_scratch, test_loader)
print(f'Test ACC: {test_acc_scratch:.4f}%')

Test ACC: 21.5000%


### 3.2 Fine-Tuning a Pretrained ResNet-18


#### Loading and Configuring the Pretrained Model


In [11]:
model_ft = models.resnet18(weights="IMAGENET1K_V1")
model_ft.fc = nn.Linear(in_features, num_classes)
criterion = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer_ft = optim.Adam(model_ft.parameters(), lr=0.001)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s]


#### Training the Fine-Tuned Model


In [12]:
model_ft = train_model(model=model_ft,
                       criterion = criterion,
                       trainloader = train_loader,
                       validateloader = val_loader,
                       optimizer=optimizer_ft,
                       scheduler = exp_lr_scheduler,
                       num_epochs=25,
                       device=device)

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 1 | Loss: 1.3638 | Train Acc: 49.1000% | Val Acc: 46.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.5188 | Train Acc: 71.5000% | Val Acc: 52.0000%


...

Epoch 23 | Loss: 0.0018 | Train Acc: 100.0000% | Val Acc: 77.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 24 | Loss: 0.0018 | Train Acc: 100.0000% | Val Acc: 78.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 25 | Loss: 0.0021 | Train Acc: 100.0000% | Val Acc: 78.0000%
Training complete in 3m 5s
Best val Acc: 79.0000%


#### Evaluating the Fine-Tuned Model


In [13]:
test_acc_ft = validate_model(model_ft, test_loader)
print(f'Test ACC: {test_acc_ft:.4f}%')

Test ACC: 71.0000%


### 3.3 Fixed Feature Extraction (Linear Probing)


#### Freezing Pretrained Layers and Replacing the Classifier


In [14]:
model_conv = models.resnet18(weights="IMAGENET1K_V1")
for param in model_conv.parameters():
    param.requires_grad = False
model_conv.fc = nn.Linear(in_features, num_classes)
criterion = nn.CrossEntropyLoss()

# Observe that only parameters of final layer are being optimized as
# opposed to before.
optimizer_conv = optim.Adam(model_conv.fc.parameters(), lr=0.01)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_conv, step_size=7, gamma=0.1)

#### Training the Classification Layer


In [15]:
model_conv = train_model(model=model_conv,
                       criterion = criterion,
                       trainloader = train_loader,
                       validateloader = val_loader,
                       optimizer=optimizer_conv,
                       scheduler = exp_lr_scheduler,
                       num_epochs=25,
                       device=device)

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 1 | Loss: 2.9662 | Train Acc: 23.6000% | Val Acc: 24.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 2 | Loss: 1.4749 | Train Acc: 60.4000% | Val Acc: 60.0000%


...

Epoch 23 | Loss: 0.3785 | Train Acc: 91.6000% | Val Acc: 73.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 24 | Loss: 0.3786 | Train Acc: 91.7000% | Val Acc: 72.0000%


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 25 | Loss: 0.3763 | Train Acc: 91.6000% | Val Acc: 73.0000%
Training complete in 2m 19s
Best val Acc: 75.0000%


#### Evaluating the Fixed Feature Extractor


In [16]:
test_acc_conv = validate_model(model_conv, test_loader)
print(f'Test ACC: {test_acc_conv:.4f}%')

Test ACC: 67.5000%


## 4. Test Accuracy Comparison


#### Comparing the Three Methods


In [17]:
print(f'From Scratch: {test_acc_scratch:.4f}%')
print(f'Fine-Tuning: {test_acc_ft:.4f}%')
print(f'Fixed Feature Extraction: {test_acc_conv:.4f}%')

From Scratch: 21.5000%
Fine-Tuning: 71.0000%
Fixed Feature Extraction: 67.5000%
